In [1]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")


🚀 Environment: Local
📁 Working directory set to: /home/aidmantas/repos/computer-data-analysis-report
✅ Environment ready!


# Data Merging: Unified AI Data Center Dataset

This notebook combines multiple raw data sources into a single, clean feature matrix for machine learning model training.

In [2]:
import pandas as pd
import numpy as np

# Load data
promises = pd.read_csv('data/raw/buildout_promises_expanded.csv')
financials = pd.read_csv('data/raw/company_financials_expanded.csv')
ratios = pd.read_csv('data/raw/company_financial_ratios.csv')
grid = pd.read_csv('data/raw/grid_interconnection_queue.csv')
macro = pd.read_csv('data/raw/macro_economic_indicators.csv')

# FIX: Cleanup ticker column (some values like 'TICKER_MSFT' need mapping back to 'MSFT')
promises['ticker_clean'] = promises['company_ticker'].str.replace('TICKER_', '')
# If columns are shifted due to CSV parsing, ensure we use the correct one
if (promises['ticker_clean'] == 'CAISO').any():
    # Shift detected (likely due to quoted location with comma)
    promises['ticker_clean'] = promises['company_tier'].str.replace('TICKER_', '')

# Convert dates
promises['announcement_date'] = pd.to_datetime(promises['announcement_date'])
macro['date'] = pd.to_datetime(macro['date'], format='ISO8601', utc=True).dt.tz_localize(None)
grid['Queue Date'] = pd.to_datetime(grid['Queue Date'], errors='coerce')
financials['Date'] = pd.to_datetime(financials['Date'], utc=True).dt.tz_localize(None)

print(f"Loaded {len(promises)} promises, {len(financials)} financial rows.")

Loaded 34 promises, 10048 financial rows.


### 1. Macro Economic Merging (Quarterly Resampling)

In [3]:
try:
    macro_q = macro.set_index('date').resample('QE').last().reset_index()
except:
    macro_q = macro.set_index('date').resample('Q').last().reset_index()

macro_q['quarter'] = macro_q['date'].dt.to_period('Q')
promises['announcement_quarter'] = promises['announcement_date'].dt.to_period('Q')

merged_df = promises.merge(macro_q.drop(columns=['date']), left_on='announcement_quarter', right_on='quarter', how='left')
merged_df = merged_df.drop(columns=['quarter'])

print(f"After Macro Merge: {merged_df.shape}")

After Macro Merge: (34, 23)


### 2. Company Financials (Quarterly Resampling)

In [4]:
financials['quarter'] = financials['Date'].dt.to_period('Q')

# Select numeric columns for aggregation
agg_cols = ['Close', 'Volume', 'market_cap', 'revenue', 'ebitda']
available_cols = [c for c in agg_cols if c in financials.columns]

fin_q = financials.groupby(['ticker', 'quarter'])[available_cols].mean().reset_index()
fin_q = fin_q.rename(columns={c: f"q_{c.lower()}" for c in available_cols})

merged_df = merged_df.merge(fin_q, left_on=['ticker_clean', 'announcement_quarter'], right_on=['ticker', 'quarter'], how='left')
merged_df = merged_df.drop(columns=['ticker', 'quarter'])

print(f"After Financials Merge: {merged_df.shape}")

After Financials Merge: (34, 25)


### 3. Static Financial Ratios

In [5]:
ratio_cols = ['ticker', 'debt_to_equity', 'current_ratio', 'quick_ratio', 'return_on_assets', 'return_on_equity', 'profit_margin']
available_ratios = [c for c in ratio_cols if c in ratios.columns]
merged_df = merged_df.merge(ratios[available_ratios], left_on='ticker_clean', right_on='ticker', how='left')
merged_df = merged_df.drop(columns=['ticker'])

print(f"After Ratios Merge: {merged_df.shape}")

After Ratios Merge: (34, 31)


### 4. Grid Interconnection Queue Features

In [6]:
grid['quarter'] = grid['Queue Date'].dt.to_period('Q')
grid_counts = grid.groupby(['iso', 'quarter']).size().reset_index(name='iso_queue_depth')

merged_df = merged_df.merge(grid_counts, left_on=['iso', 'announcement_quarter'], right_on=['iso', 'quarter'], how='left')
merged_df['iso_queue_depth'] = merged_df['iso_queue_depth'].fillna(0)
merged_df = merged_df.drop(columns=['quarter'])

print(f"After Grid Merge: {merged_df.shape}")

After Grid Merge: (34, 32)


### 5. Final Dataset

In [7]:
os.makedirs('data/processed', exist_ok=True)
merged_df.to_csv('data/processed/dataset_for_ml.csv', index=False)
print(f"🎉 Unified dataset saved: {len(merged_df)} rows, {len(merged_df.columns)} features")

🎉 Unified dataset saved: 34 rows, 32 features
